# 同任务的单 Worker、并行 Worker 与确定性基线

> 状态：verified；教学数据；执行日期：2026-09-06。

先读[案例](../03-cases/01-single-vs-multi-agent.md)。这个实验验证调度、合并与失败，不调用真实语言模型。输入已经具备全部字段，所以确定性程序是必须保留的基线。异步等待只模拟 I/O，不是推理耗时的业务测量。

目标：在 latency_ms≤20、memory_mb≤220 的候选内按 quality 最高值选择。

执行说明：本次因环境禁止 Kernel socket，使用独立 Python 进程内 IPython 按单元顺序执行并保存真实输出；不是 Jupyter kernel 运行。普通 Jupyter 环境可按原顺序运行。

In [1]:
from pathlib import Path
import sys, json
from pprint import pprint
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "10-Knowledge").is_dir())
sys.path.insert(0, str(ROOT / "10-Knowledge/08-planning-workflow-multi-agent/05-code/multi-agent-runtime-python/src"))
from multi_agent import Task, WorkerResult, Router, Supervisor, merge_results, MergeConflict
from multi_agent.fixture import CANDIDATES, experiment, choose
import asyncio
pprint(CANDIDATES)


[{'latency_ms': 18, 'memory_mb': 200, 'name': 'A', 'quality': 0.91},
 {'latency_ms': 29, 'memory_mb': 150, 'name': 'B', 'quality': 0.93},
 {'latency_ms': 12, 'memory_mb': 90, 'name': 'C', 'quality': 0.88}]


先根据输入手算：B 的质量最高，但时延 29 ms 超过 20 ms，所以只在 A/C 中选择，答案是 A。下面两种 Worker 模式使用同一份代码和验收规则，差别只有并发数。

In [2]:
evidence = await experiment()
pprint(evidence["rows"])
assert [r["selected"] for r in evidence["rows"]] == ["A", "A", "A"]
assert [r["max_active"] for r in evidence["rows"][:2]] == [1, 2]
print("并行实际事件：")
pprint(evidence["traces"]["two_workers_parallel"])


[{'calls': 2,
  'elapsed_ms': 40.577,
  'max_active': 1,
  'mode': 'single_worker_serial',
  'selected': 'A'},
 {'calls': 2,
  'elapsed_ms': 20.292,
  'max_active': 2,
  'mode': 'two_workers_parallel',
  'selected': 'A'},
 {'calls': 0,
  'max_active': 0,
  'mode': 'deterministic_baseline',
  'selected': 'A'}]
并行实际事件：
[{'elapsed_ms': 0.024, 'event': 'queued', 'seq': 0, 'task_id': 'quality'},
 {'elapsed_ms': 0.037,
  'event': 'started',
  'kind': 'quality',
  'seq': 1,
  'task_id': 'quality'},
 {'elapsed_ms': 0.097, 'event': 'queued', 'seq': 2, 'task_id': 'constraints'},
 {'elapsed_ms': 0.104,
  'event': 'started',
  'kind': 'constraints',
  'seq': 3,
  'task_id': 'constraints'},
 {'elapsed_ms': 20.27,
  'event': 'completed',
  'keys': ['quality'],
  'seq': 4,
  'task_id': 'quality'},
 {'elapsed_ms': 20.292,
  'event': 'completed',
  'keys': ['feasible'],
  'seq': 5,
  'task_id': 'constraints'}]


串行存在两个 started/completed 区间，且最大活跃数为 1；并行最大活跃数为 2。耗时随运行环境改变，不对速度比设置硬阈值。两种模式质量相同，不能据此声称多 Agent 能提高答案质量。

接下来模拟约束评审超时。质量分数仍应保留，但最终选择必须停止。

In [3]:
async def good(task): return {"quality": {"A": 0.91}}
async def slow(task):
    try:
        await asyncio.sleep(10)
        return {"feasible": ["A"]}
    finally:
        cleanup.append("slow cleaned")
cleanup = []
partial = await Supervisor(Router({"good": good, "slow": slow})).run([
    Task("q", "good", {}, ("quality",)),
    Task("c", "slow", {}, ("feasible",), timeout_s=0.02),
])
merged = merge_results(partial.results, expected_task_ids=["q", "c"])
pprint({"values": merged.values, "missing_tasks": merged.missing_tasks, "cleanup": cleanup})
assert not merged.complete and cleanup == ["slow cleaned"]
try:
    choose(merged)
except ValueError as exc:
    print("正确拒绝不完整决策：", exc)
else:
    raise AssertionError("incomplete review was accepted")
evidence["partial_failure_trace"] = partial.trace


{'cleanup': ['slow cleaned'],
 'missing_tasks': ['c'],
 'values': {'quality': {'A': 0.91}}}
正确拒绝不完整决策： cannot decide with missing review


再验证冲突不会被后返回的结果覆盖。相同字段的不同值需要显式处理；模型喜欢哪一个不能取代证据。

In [4]:
try:
    merge_results([WorkerResult("review-A", "ok", {"decision": "A"}),
                   WorkerResult("review-B", "ok", {"decision": "B"})])
except MergeConflict as exc:
    print("正确暴露冲突：", exc)
    evidence["conflict"] = str(exc)
else:
    raise AssertionError("conflict was silently overwritten")


正确暴露冲突： decision: ['review-A'] conflicts with review-B


取消父任务之后，子任务必须清理并退出。这里只能证明合作式协程取消；阻塞 CPU 或不理会取消的外部服务需要另外的执行边界。

In [5]:
started, stopped = asyncio.Event(), asyncio.Event()
async def cancellable(task):
    started.set()
    try:
        await asyncio.sleep(10)
    finally:
        stopped.set()
supervisor = Supervisor(Router({"slow": cancellable}))
parent = asyncio.create_task(supervisor.run([Task("cancel-me", "slow", {}, (), timeout_s=20)]))
await started.wait()
parent.cancel()
try:
    await parent
except asyncio.CancelledError:
    print("父任务取消已传播；子任务已清理：", stopped.is_set())
assert stopped.is_set()
evidence["cancel_trace"] = supervisor.last_trace
artifact = ROOT / "10-Knowledge/08-planning-workflow-multi-agent/04-labs/artifacts/experiment.json"
artifact.parent.mkdir(exist_ok=True)
artifact.write_text(json.dumps(evidence, ensure_ascii=False, indent=2), encoding="utf-8")
print("证据已保存：", artifact)


父任务取消已传播；子任务已清理： True
证据已保存： /workspace/scratch/46290c6cc5b4/TheBestAIGuide/10-Knowledge/08-planning-workflow-multi-agent/04-labs/artifacts/experiment.json


本实验确认了相同答案、并发上界、部分失败拒绝、冲突暴露与父子取消。没有评估真实模型、跨进程隔离或分布式恢复。

练习：把候选 B 的 latency_ms 改成 19，解释为何三个基线都应该改选 B；再把质量 Worker 的返回键改成 `score`，观察契约校验在哪里失败。完整实现见[工程说明](../05-code/multi-agent-runtime-python/README.md)。